# QuickSelect: al k-lea cel mai mic element

Problema: dat un vector nesortat, găsiți al **k-lea** cel mai mic element (k = 1 → minimul, k = n → maximul, k = ⌈n/2⌉ → **mediana**).

- Cu sortare: O(n log n).
- Minimul sau maximul: O(n), o singură parcurgere.
- **QuickSelect**: O(n) în medie. Partiționăm ca la QuickSort, dar continuăm **doar în partea care conține răspunsul**.
- **Mediana medianelor** (Blum, Floyd, Pratt, Rivest, Tarjan, 1973): O(n) **în cazul cel mai nefavorabil**.

## 1. QuickSelect

După partiționare, pivotul este pe poziția `pos`, adică locul lui final. Dacă `pos` este exact poziția căutată, am terminat. Altfel continuăm **doar** în stânga sau **doar** în dreapta. Avem un singur apel recursiv, așa că algoritmul se scrie ușor iterativ.

In [ ]:
import random


# pivot aleator, il mutam la final si facem lomuto normal
def partition_lomuto(a, lo, hi):
    r = random.randint(lo, hi)
    a[r], a[hi] = a[hi], a[r]
    p = a[hi]
    i = lo
    for j in range(lo, hi):
        if a[j] < p:
            a[i], a[j] = a[j], a[i]
            i += 1
    a[i], a[hi] = a[hi], a[i]
    return i


# al k-lea cel mai mic element, k de la 1 la n
def quickselect(arr, k):
    assert 1 <= k <= len(arr)
    a = list(arr)                 # copie, ca sa nu stricam vectorul primit
    target = k - 1                # pozitia lui in vectorul sortat
    lo, hi = 0, len(a) - 1
    while True:
        pos = partition_lomuto(a, lo, hi)
        if pos == target:
            return a[pos]
        if target < pos:
            hi = pos - 1          # raspunsul e in stanga
        else:
            lo = pos + 1          # raspunsul e in dreapta


v = [7, 10, 4, 3, 20, 15, 8, 1, 12]
print("sortat:", sorted(v))
for k in [1, 3, 5, 9]:
    print(f"k = {k}: {quickselect(v, k)}")

Pas cu pas: urmărim cum se micșorează intervalul `[lo, hi]`.

In [ ]:
def quickselect_verbose(arr, k):
    a = list(arr)
    target = k - 1
    lo, hi = 0, len(a) - 1
    while True:
        pos = partition_lomuto(a, lo, hi)
        print(f"[{lo}, {hi}]: pivotul {a[pos]} ajunge pe poziția {pos}  ->  {a[lo:hi + 1]}")
        if pos == target:
            return a[pos]
        if target < pos:
            hi = pos - 1
        else:
            lo = pos + 1


random.seed(1)
quickselect_verbose([7, 10, 4, 3, 20, 15, 8, 1, 12], 5)

## 2. Complexitate

- **Cazul favorabil**: pivotul ajunge la mijloc ⇒ $T(n) = T(n/2) + O(n)$ ⇒ $n + \frac{n}{2} + \frac{n}{4} + \dots \le 2n$ ⇒ **O(n)**
- **Cazul mediu** (pivot aleator): **O(n)**
- **Cazul nefavorabil**: pivotul e mereu minimul sau maximul ⇒ $T(n) = T(n-1) + O(n)$ ⇒ **O(n²)**

Diferența față de QuickSort: acolo recursivitatea se face pe **ambele** părți ($2T(n/2)$ ⇒ n log n), aici doar pe **una** ($T(n/2)$ ⇒ n).

Verificăm: numărul de comparații împărțit la n rămâne aproximativ constant.

In [ ]:
def quickselect_comparisons(arr, k):
    a = list(arr)
    target, lo, hi = k - 1, 0, len(a) - 1
    comparisons = 0
    while True:
        comparisons += hi - lo
        pos = partition_lomuto(a, lo, hi)
        if pos == target:
            return comparisons
        if target < pos:
            hi = pos - 1
        else:
            lo = pos + 1


for n in [1_000, 10_000, 100_000]:
    v = random.sample(range(10 * n), n)
    c = quickselect_comparisons(v, n // 2)
    print(f"n = {n:>7}: {c:>8} comparații  ->  {c / n:.2f} · n")

## 3. Mediana medianelor: O(n) în cazul cel mai nefavorabil

Problema la QuickSelect este un pivot prost. Soluția: alegem un pivot **garantat bun**.

1. Împărțim vectorul în grupe de câte **5** elemente.
2. Sortăm fiecare grupă (5 elemente ⇒ O(1)) și luăm mediana ei ⇒ ⌈n/5⌉ mediane.
3. **Recursiv**, găsim mediana acestor mediane. Ea este pivotul `x`.
4. Partiționăm după `x` și continuăm ca la QuickSelect.

De ce e bun pivotul? Jumătate din cele n/5 grupe au mediana ≤ x, iar în fiecare astfel de grupă cel puțin 3 elemente sunt ≤ x (mediana și cele 2 mai mici decât ea):
$$\#\{\text{elemente} \le x\} \ge 3 \cdot \frac{n}{10} = \frac{3n}{10}$$
La fel, cel puțin 3n/10 elemente sunt ≥ x. Deci partea în care continuăm are **cel mult 7n/10** elemente.

$$T(n) \le \underbrace{T\!\left(\tfrac{n}{5}\right)}_{\text{mediana medianelor}} + \underbrace{T\!\left(\tfrac{7n}{10}\right)}_{\text{continuarea}} + O(n)$$

Cum $\frac{1}{5} + \frac{7}{10} = \frac{9}{10} < 1$, costul pe niveluri scade geometric: $cn\left(1 + \tfrac{9}{10} + \left(\tfrac{9}{10}\right)^2 + \dots\right) = 10\,cn$ ⇒ **T(n) = O(n)**.

(Grupele incomplete adaugă mici corecții constante, care nu schimbă rezultatul.)

Pentru claritate, partiționarea de mai jos construiește 3 liste noi (`< x`, `== x`, `> x`), deci nu e in-place. Tot O(n) rămâne.

In [ ]:
# al k-lea cel mai mic element, O(n) si in cel mai rau caz
def select_mom(a, k):
    if len(a) <= 5:
        return sorted(a)[k - 1]

    # 1-2. medianele grupelor de cate 5
    groups = [sorted(a[i:i + 5]) for i in range(0, len(a), 5)]
    medians = [g[len(g) // 2] for g in groups]

    # 3. mediana medianelor, recursiv
    x = select_mom(medians, (len(medians) + 1) // 2)

    # 4. partitionare in 3 dupa x
    less = [e for e in a if e < x]
    equal = [e for e in a if e == x]
    greater = [e for e in a if e > x]

    if k <= len(less):
        return select_mom(less, k)
    if k <= len(less) + len(equal):
        return x
    return select_mom(greater, k - len(less) - len(equal))


v = [7, 10, 4, 3, 20, 15, 8, 1, 12]
print([select_mom(v, k) for k in range(1, len(v) + 1)])
print(sorted(v))

Grupele și medianele pentru un vector de 25 de elemente distincte. Avem 5 grupe; 3 dintre ele au mediana ≤ x și contribuie fiecare cu cel puțin 3 elemente ≤ x, deci cel puțin 9 elemente sunt ≤ x (la fel pentru ≥ x):

In [ ]:
random.seed(7)
v = random.sample(range(100), 25)
groups = [sorted(v[i:i + 5]) for i in range(0, len(v), 5)]
for g in groups:
    print(g, "-> mediana", g[2])
medians = [g[2] for g in groups]
x = select_mom(medians, 3)
print("\nmedianele sortate:", sorted(medians), "-> mediana medianelor x =", x)
print(f"elemente <= x: {sum(e <= x for e in v)},  elemente >= x: {sum(e >= x for e in v)}  (garantat: fiecare >= 9)")

### De ce grupe de 5?

Cu grupe de **3** avem cel puțin 2 · (n/6) = n/3 elemente de fiecare parte, deci continuăm pe cel mult 2n/3:
$$T(n) = T\!\left(\tfrac{n}{3}\right) + T\!\left(\tfrac{2n}{3}\right) + O(n), \qquad \tfrac13 + \tfrac23 = 1 \;\Rightarrow\; O(n \log n)$$
Costul nu mai scade de la un nivel la altul. Cu grupe de 5 (sau 7, 9, ...) suma e < 1, deci O(n).

În practică mediana medianelor are o constantă mare, așa că QuickSelect cu pivot **aleator** e de obicei mai rapid. `std::nth_element` din C++ este tot o variantă de QuickSelect (cu optimizări). Comparația de timp este în `quickselect.cpp`.

## Testare

In [ ]:
for _ in range(300):
    arr = [random.randint(-30, 30) for _ in range(random.randint(1, 60))]
    s = sorted(arr)
    for k in range(1, len(arr) + 1):
        assert quickselect(arr, k) == s[k - 1]
        assert select_mom(arr, k) == s[k - 1]
print("toate testele au trecut")